In [56]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch
import time
import pandas as pd

In [7]:
model = 'Qwen/Qwen2.5-0.5B'
tokenizer = AutoTokenizer.from_pretrained(model)
qwen = AutoModel.from_pretrained(model)
causal_lm = AutoModelForCausalLM.from_pretrained(model)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
qwen = qwen.to(device)
causal_lm = causal_lm.to(device)

In [84]:
def generate_response(prompt, strategy, model, causal_lm, tokenizer, n=100, T=1, k=None, p=None, eos_id=None):
    """
    Enter a prompt to generate the next tokens in a loop until maximum token limit or end of sequence or user interruption occurs.\n
    model = GPT decoder model\n
    tokenizer = tokenizer same as the model, mixing different tokenizer might result in broken text generation\n
    strategy = ["Greedy", "Temperature", "Top-k", "Top-p"]\n
    n = number of tokens, T = temperature, k = k in top-k, p = p in top-p\n
    eos_id -> end of sequence id used to stop the text generation on reaching end of sequence. If set None the model keeps on generating text until it reaches the maximum token limit.\n
    """

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    temperature=T
    eos = eos_id
    past_key_values = None
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    generated_ids = input_ids.clone()
    prompt_tokens = len(tokenizer(prompt).input_ids)

    if strategy=='Greedy':
        start = time.perf_counter()
        for x in range(n):
            with torch.no_grad():
                base_output = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
                hidden_states = base_output.last_hidden_state
                base_logits = causal_lm.lm_head(hidden_states)
                past_key_values = base_output.past_key_values

            scaled_logits = base_logits[:, -1, :]/temperature
            probs = torch.softmax(scaled_logits, dim=-1)

            next_token_id = torch.argmax(probs ,dim=-1, keepdim=True)
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)
            if x==0:
              ttfs = time.perf_counter() - start

            if next_token_id==eos:
                break

            next_token = tokenizer.decode(next_token_id)
            input_ids = next_token_id

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)
        total_time=time.perf_counter() - start

    elif strategy=="Temperature":
        start = time.perf_counter()
        for x in range(n):
            with torch.no_grad():
                base_output = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
                hidden_state = base_output.last_hidden_state
                base_logits = causal_lm.lm_head(hidden_state)
                past_key_values = base_output.past_key_values

            scaled_logits = base_logits[:, -1, :]/temperature
            probs = torch.softmax(scaled_logits, dim=-1)

            next_token_id = torch.multinomial(probs, num_samples=1)
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)
            if x==0:
              ttfs = time.perf_counter() - start

            if next_token_id==eos:
                break

            input_ids = next_token_id

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)
        total_time=time.perf_counter() - start

    elif strategy=="Top-k":
        start = time.perf_counter()
        for x in range(n):
            with torch.no_grad():
                bo = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
                hs = bo.last_hidden_state
                base_logits = causal_lm.lm_head(hs)
                past_key_values = base_output.past_key_values

            logits = base_logits[:, -1, :]

            scaled_logits = logits/temperature

            values, indices = torch.topk(scaled_logits, k)
            probs = torch.softmax(values, dim=-1)

            sample = torch.multinomial(probs, 1)
            next_token_id = indices.gather(-1, sample)
            input_ids = next_token_id
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)
            if x==0:
              ttfs = time.perf_counter() - start

            if next_token_id==eos:
                break

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)
        total_time=time.perf_counter() - start

    elif strategy=="Top-p":
        start = time.perf_counter()
        for x in range(n):
            with torch.no_grad():
                base_output = model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)
                hs = base_output.last_hidden_state
                base_logits = causal_lm.lm_head(hs)
                past_key_values = base_output.past_key_values

            logits = base_logits[:, -1, :]
            scaled_logits = logits/temperature
            sorted_logits, sorted_indices = torch.sort(scaled_logits, descending=True)

            sorted_probs = torch.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
            mask = cumulative_probs<=p
            mask[..., 0] = True

            filtered_probs = sorted_probs * mask
            filtered_probs /= filtered_probs.sum(dim=-1)

            sample = torch.multinomial(filtered_probs, num_samples=1)

            next_token_id = sorted_indices.gather(-1, sample)
            input_ids = next_token_id
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)
            if x==0:
              ttfs = time.perf_counter() - start

            if next_token_id == eos:
                break

            new_text = tokenizer.decode(next_token_id[0])
            print(new_text, end="", flush=True)
        total_time=time.perf_counter() - start

    else:
        print("Invalid strategy. Enter a valid strategy to generate tokens.")

    return {
        "total_time_taken": total_time,
        "ttfs": ttfs,
        "prompt_tokens": prompt_tokens,
        "generated_tokens": generated_ids.shape[1]-prompt_tokens
    }

## Variable Output Lengths

In [85]:
output_lengths = [20, 100, 300, 500]

record_greedy = {}
for x in output_lengths:
    response = generate_response(
        prompt="Explain about Harry Potter and the Philosopher's Stone.",
        strategy="Greedy",
        model=qwen,
        causal_lm=causal_lm,
        tokenizer=tokenizer,
        n=x,
        eos_id=None
    )
    record_greedy[x] = response
    print()

 The Philosopher's Stone is a book written by J.K. Rowling, the author of the Harry
 The Philosopher's Stone is a book written by J.K. Rowling, the author of the Harry Potter series. It is the first book in the series and is the first book in the Harry Potter universe. The book is a magical book that contains the power to turn any person into a wizard. The book is written in a language that is not understood by the wizarding world, and it is the first book in the series that is not a part of the Harry Potter universe. The book is
 The Philosopher's Stone is a book written by J.K. Rowling, the author of the Harry Potter series. It is the first book in the series and is the first book in the Harry Potter universe. The book is a magical book that contains the power to turn any person into a wizard. The book is written in a language that is not understood by the wizarding world, and it is the first book in the series that is not a part of the Harry Potter universe. The book is also the fir

In [86]:
df = pd.DataFrame(record_greedy.items(), columns=["Output Tokens", "response"])

for x, y in df['response'].items():
    df.loc[x, 'total_time_taken'] = y['total_time_taken']
    df.loc[x, 'ttfs'] = y['ttfs']
    df.loc[x, 'prompt_tokens'] = y['prompt_tokens']

df.drop(columns=['response'], inplace=True)

df['tokens/s'] = df['Output Tokens']/df['total_time_taken']
df

,Output Tokens,total_time_taken,ttfs,prompt_tokens,tokens/s
0,20,2.604164,0.141294,12.0,7.680007
1,100,6.989719,0.217428,12.0,14.306728
2,300,22.840267,0.086249,12.0,13.134698
3,500,30.865356,0.058956,12.0,16.199392


## Variable Prompt Length

In [87]:
base_prompt = (
    "Explain the process of photosynthesis in detail, including the light-dependent "
    "reactions, the Calvin cycle, the role of chlorophyll, ATP, NADPH, carbon dioxide, "
    "glucose production, and its importance for ecosystems."
)

def make_prompt(base_prompt, tokenizer, target_tokens):
    prompt = base_prompt
    while len(tokenizer(prompt).input_ids) < target_tokens:
        prompt += " " + base_prompt
    return prompt

In [88]:
prompt_20 = make_prompt(base_prompt, tokenizer, 20)
prompt_200 = make_prompt(base_prompt, tokenizer, 200)
prompt_500 = make_prompt(base_prompt, tokenizer, 500)
prompt_1000 = make_prompt(base_prompt, tokenizer, 1000)
prompt_2000 = make_prompt(base_prompt, tokenizer, 2000)

In [89]:
prompts = [prompt_20, prompt_200, prompt_500, prompt_1000, prompt_2000]

record_greedy_prompt = {}
for x in prompts:
    response = generate_response(
        prompt=x,
        strategy="Greedy",
        model=qwen,
        causal_lm=causal_lm,
        tokenizer=tokenizer,
        n=150,
        eos_id=None
    )
    record_greedy_prompt[x] = response
    print()

 Provide examples of plants and animals that use photosynthesis for energy and growth. Photosynthesis is the process by which plants, algae, and some bacteria convert light energy into chemical energy in the form of glucose. The process involves several stages, including the light-dependent reactions, the Calvin cycle, and the role of chlorophyll, ATP, NADPH, carbon dioxide, glucose production, and its importance for ecosystems.

Light-dependent reactions occur in the thylakoid membranes of chloroplasts. These reactions involve the absorption of light energy by chlorophyll, which then converts it into chemical energy in the form of ATP and NADPH. The light-dependent reactions are responsible for the production of glucose, which is the primary source of energy for plants
 Explain the process of photosynthesis in detail, including the light-dependent reactions, the Calvin cycle, the role of chlorophyll, ATP, NADPH, carbon dioxide, glucose production, and its importance for ecosystems. Ex

In [96]:
df = pd.DataFrame(record_greedy_prompt.items(), columns=["Prompt Tokens", "response"])

for x, y in df['response'].items():
    df.loc[x, 'prompt_tokens'] = y['prompt_tokens']
    df.loc[x, 'generated_tokens'] = y['generated_tokens']
    df.loc[x, 'total_time_taken'] = y['total_time_taken']
    df.loc[x, 'ttft'] = y['ttfs']

df.drop(columns=['response', 'Prompt Tokens'], inplace=True)

df['tokens/s'] = df['generated_tokens']/df['total_time_taken']
df

,prompt_tokens,generated_tokens,total_time_taken,ttft,tokens/s
0,45.0,150.0,6.366075,0.039247,23.562398
1,221.0,150.0,5.740459,0.051876,26.130315
2,529.0,150.0,8.960048,0.079548,16.740982
3,1013.0,150.0,6.418427,0.148616,23.370211
4,2025.0,150.0,8.845335,0.293422,16.958092
